In [ ]:
from firebase_util import FirestoreDocumentValidator
from pydantic import BaseModel, Field, ConfigDict, field_validator
# from google.api_core.datetime_helpers import DatetimeWithNanoseconds
from datetime import datetime
from typing_extensions import Self
from typing import Any
import re
from urllib.parse import urlparse
from copy import deepcopy

PROJECT_ID = "wilde-rosen-483d1"
# PROJECT_ID = "wilde-rosen-npr-dfafc"

# Set up validator
validator = FirestoreDocumentValidator(PROJECT_ID)

In [ ]:
schema = validator.guess_schema_from_doc("cars/kangoo/damages/AAss1HaFcfgKAI7WHkTs")
for k, v in sorted(schema.items()):
    print(f"{k}: {v}")

In [ ]:
docs = validator.list_documents_in_collection("cars/kangoo/damages")

In [ ]:
# Validate fields/types in the destination documents
validator.verify_document_fields(docs, schema)

In [ ]:
def url_to_path(url: str) -> str | None:
    regex = r'(https:\/\/firebasestorage\.googleapis\.com\/v0\/b\/)(.*)(\.firebasestorage\.app\/o\/)([^\?]*)(\??)(.*)'
    match = re.match(regex, url)
    if match is not None and len(match.groups()) >= 4:
        return match.group(4)
    return None


class DamageDetail(BaseModel):
    imagePath: str = Field(validation_alias="path")
    description: str
    imageUrl: str = ""

    model_config = ConfigDict(validate_by_alias=True, validate_by_name=True)

    @model_validator(mode='before')
    @classmethod
    def check_card_number_not_present(cls, data: Any) -> Any:
        if isinstance(data, dict):
            if "imagePath" not in data or "path" not in data and "imageUrl" in data:
                path = url_to_path(data["imageUrl"])
                if path is not None:
                    d = deepcopy(data)
                    d["imagePath"] = path
                    return d
        return data


class DamageEntry(BaseModel):
    createdAt: datetime
    description: str
    details: list[DamageDetail]
    imagePath: str = Field(validation_alias="path")
    order: int
    side: str
    updatedAt: datetime
    x: int
    y: int
    imageUrl: str = ""
    schematicUrl: str = ""

    model_config = ConfigDict(validate_by_alias=True, validate_by_name=True)

    @model_validator(mode='before')
    @classmethod
    def check_card_number_not_present(cls, data: Any) -> Any:
        if isinstance(data, dict):
            if "imagePath" not in data or "path" not in data and "imageUrl" in data:
                path = url_to_path(data["imageUrl"])
                if path is not None:
                    d = deepcopy(data)
                    d["imagePath"] = path
                    return d
        return data

class Car(BaseModel):
    title: str
    

In [ ]:
doc_path = "cars/kangoo/damages/AAss1HaFcfgKAI7WHkTs"
raw_data =validator.get_doc(doc_path)
# raw_data =validator.get_doc("cars/kangoo")
raw_data

In [ ]:
raw_data

In [ ]:
# raw_data['path'] = raw_data['imagePath']
del(raw_data['imagePath'])

raw_data

In [ ]:
raw_data

In [ ]:
data = DamageEntry.model_validate(raw_data)
data.model_dump()

In [ ]:
raw_data

In [ ]:
validator.set_doc(doc_path, data.model_dump())